# Chunk 30 — Clean backbone, mode targets, feature caches

Builds everything Chunks 31–33 consume. Every step is gated by `ch29_decisions.json`, excludes grid 55, and never scores test labels.

| step | what | where | writes |
|---|---|---|---|
| 1 | no-55 split manifest, checked against Chunk 28/29's evaluation sets | any | `ch30_splits.json` |
| 2a | feed-component node feature, checked against Chunk 29 | any | — |
| 2b | `best_model` → 6 inputs with a zero column; float64 exactness guard | CPU | — |
| 2c | legacy fine-tune on the 35+45 pool (Chunk 22 L0 recipe) | **GPU** | `checkpoints/backbone_no55.pt` |
| 2d | pre-registered gate: `POOL_FINETUNE_VAL` MSE ≤ 1.35 dB² | **GPU** | `ch30_backbone_report.json` |
| 3 | K = 3 mode targets, scipy, sharded, resumable | **CPU** | `ch30_mode_targets{,_ft}.parquet` |
| 4 | readout and node-embedding cache (only if the gate passed) | **GPU** | `ch30_readout.npy`, `ch30_node_emb_f16.dat`, … |
| 5 | V2 eigenvalue cache for every graph | **CPU** | `ch30_eigs.parquet` |

**Two-session plan.** Run header → Setup → Staging → Step 1 → Step 2 → Step 4 on the A100.
Steps 3 and 5 are standalone: in a **second, CPU runtime**, run header → Setup → Staging → Step 3 → Step 5.
They read `ch30_splits.json`, so run Step 1 first. Step 3 is the long one (~0.72 s per curve per CPU, about 100k curves). It fits the 35/45 curves first, so Chunk 31 can start before the 25×25 curves finish. Shards make it resumable across sessions.

## Cell 1 — Installs

In [ ]:
!pip -q install torch_geometric pyarrow

## Cell 2 — Repository and Drive
Clones or pulls the repo, puts `src/` on the path, and mounts Drive.

In [ ]:
import os, sys, subprocess
REPO_ROOT = '/content/antenna_gnn'
if os.path.isdir(f'{REPO_ROOT}/.git'):
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/asparagusD/antenna_gnn.git', REPO_ROOT], check=True)
sys.path.insert(0, f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA = '/content/drive/MyDrive/antenna_dataset'
print('repo:', REPO_ROOT, '| data:', DATA_ROOT)

## Setup — decision contract and bookkeeping
**What it does:**
- Asserts the Chunk 29 decisions: VIABLE, K = 3, quadratic baseline, −60 dB floor, V2, mirror off, `LABEL_MIN`.
- Captures the mtimes of every `ch28_*`/`ch29_*` artifact and the SHA-256 of the s11 statistics, for the final guard.
- Loads the s11 statistics read-only and disables TF32.
- Defines `build_gnn()` with every constructor argument explicit.

**Writes:** nothing.

In [ ]:
# ── Setup: decision contract, constants, provenance bookkeeping ──
import json, glob, math, time, hashlib, shutil, zipfile, copy
import multiprocessing as mp
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from model import AntennaGNN
from feed_component import (PIXEL_SIZE_MM, feed_rc, feed_component_mask, feed_distance_mm,
                            AppendFeedComponentFeature)
from spectral import eig_v2_features, EIG_COLS, M_EIG
from synth import (F_GHZ, fit_curve_scipy, mode_rows_from_fit, check_numpy_torch_agreement,
                   synth_db_numpy)

ART, CKPT, FIG = f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints', f'{DATA_ROOT}/figures'
for p in (ART, CKPT, FIG):
    os.makedirs(p, exist_ok=True)
GRIDS, FT_GRIDS = (25, 35, 45), (35, 45)

# Chunk 29 decisions are binding
DEC29 = json.load(open(f'{ART}/ch29_decisions.json'))
EXPECTED = {'cell_a_branch': 'VIABLE', 'K_slots': 3, 'baseline_order': 2, 'synth_floor_db': -60,
            'spectral_variant': 'V2', 'mirror_augmentation': False, 'label': 'LABEL_MIN'}
for k, v in EXPECTED.items():
    assert DEC29.get(k) == v, f'STOP: ch29 decision {k} = {DEC29.get(k)!r}, expected {v!r}'
K_SLOTS, BASE_ORDER = DEC29['K_slots'], DEC29['baseline_order']
print('Chunk 29 decisions locked:', {k: DEC29[k] for k in EXPECTED})

# Every artifact goes through OUT(); the only non-ch30_ file allowed is the backbone checkpoint
WRITTEN = []
def OUT(name):
    assert name.startswith('ch30_'), f'refusing to write non-ch30 artifact {name}'
    p = f'{ART}/{name}'
    WRITTEN.append(p)
    return p

def file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

PRIOR_MTIMES = {p: os.path.getmtime(p) for p in glob.glob(f'{ART}/ch28_*') + glob.glob(f'{ART}/ch29_*')}
S11_SHA = {n: file_sha(f'{ART}/{n}') for n in ('s11_mean.npy', 's11_std.npy')}
print(f'Captured mtimes of {len(PRIOR_MTIMES)} ch28/ch29 artifacts and the s11 stats hashes.')

# Normalization statistics: read only, never recomputed
S11_MEAN = torch.tensor(np.load(f'{ART}/s11_mean.npy'), dtype=torch.float32)
S11_STD = torch.tensor(np.load(f'{ART}/s11_std.npy'), dtype=torch.float32)
assert S11_MEAN.shape == (201,) and S11_STD.shape == (201,)

HAS_GPU = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_GPU else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
N_CPU = os.cpu_count() or 1
print(f'Device: {DEVICE} | CPUs: {N_CPU} | TF32 off')
print('Steps 2 and 4 need the GPU. Steps 3 and 5 are CPU-only and can run in a second session.')

def build_gnn(node_feat_dim):
    return AntennaGNN(node_feat_dim=node_feat_dim, edge_feat_dim=2, hidden_dim=128, heads=8,
                      edge_dim=16, num_blocks=4, output_dim=201)
def n_params(m):
    return sum(p.numel() for p in m.parameters())
TRANSFORM = AppendFeedComponentFeature()

def _worker_init():
    # one thread per worker: avoids BLAS/OpenMP oversubscription and fork-after-threads hangs
    torch.set_num_threads(1)
    os.environ['OMP_NUM_THREADS'] = '1'

## Staging — graphs to local SSD
Unzips `data/processed/processed_{N}x{N}.zip` flat to `/content/local_graphs/{N}x{N}/sample_<i>.pt`, with a marker file so re-runs skip it. Chunk 29's runs needed this for 35/45; 25×25 is staged too, for Steps 2d, 3 and 5. It then checks that the on-disk `y` is raw dB, reading the files directly rather than through any Dataset.

In [ ]:
# ── Staging: unzip processed graphs to local SSD (flat sample_<i>.pt per grid) ──
LOCAL = '/content/local_graphs'
def stage(g):
    d = f'{LOCAL}/{g}x{g}'
    marker = f'{d}/_STAGED.txt'
    if os.path.exists(marker):
        return d
    z = f'{DATA_ROOT}/data/processed/processed_{g}x{g}.zip'
    if not os.path.exists(z):
        print(f'{g}x{g}: no zip at {z}; falling back to Drive directories')
        return None
    os.makedirs(d, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        members = [m for m in zf.namelist() if m.endswith('.pt')]
        for m in tqdm(members, desc=f'unzip {g}x{g}', leave=False):
            with zf.open(m) as src, open(f'{d}/{os.path.basename(m)}', 'wb') as dst:
                shutil.copyfileobj(src, dst)
    with open(marker, 'w') as fp:
        fp.write(f'{len(members)}\n')
    return d

STAGED = {g: stage(g) for g in GRIDS}
for g, d in STAGED.items():
    n = len(glob.glob(f'{d}/sample_*.pt')) if d else 0
    print(f'{g}x{g}: {n:,} local graphs' + ('' if d else ' (Drive fallback)'))

def graph_path(g, i):
    sub = 'processed' if g == 25 else 'processed_finetune'
    for p in (f'{LOCAL}/{g}x{g}/sample_{i}.pt',
              f'{LOCAL}/kaggle/working/data/processed/{g}x{g}/sample_{i}.pt',
              f'{DATA_ROOT}/data/{sub}/{g}x{g}/sample_{i}.pt'):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'graph {g}x{g} sample_{i}.pt not found')

def load_raw(g, i):
    return torch.load(graph_path(g, int(i)), weights_only=False)

# On-disk y must be RAW dB (identity guard on the file itself, bypassing any Dataset)
for g in GRIDS:
    some = sorted(glob.glob(f'{STAGED[g]}/sample_*.pt'))[:5] if STAGED[g] else []
    for p in some:
        y = torch.load(p, weights_only=False).y
        assert y.shape[-1] == 201 and float(y.max()) <= 0.01, f'STOP: {p} y is not raw dB (max {float(y.max()):.3f})'
print('On-disk y is raw dB.')

## Step 1 — Split manifest
**What it does:**
- Builds the splits from `indices.json` (25×25) and the three `finetune_*_indices.json` files (35/45), with grid 55 removed.
- Accepts `[grid, idx]` pairs, a dict, or bare integers for an implicit grid.
- Asserts the roles are pairwise disjoint.
- **Asserts every val and test set equals the set Chunk 28/29 scored**, taken from `ch28_per_sample.parquet`, so all earlier numbers stay comparable. v1 of this cell compared the test set with itself.

**Writes:** `ch30_splits.json`.

In [ ]:
# ── Step 1: split manifest without 55x55, checked against the Chunk 28/29 evaluation sets ──
def to_pairs(obj, default_grid=None):
    if isinstance(obj, dict):
        return {(int(g), int(i)) for g, vs in obj.items() for i in vs}
    out = set()
    for e in obj:
        if isinstance(e, (list, tuple)) and len(e) == 2:
            out.add((int(e[0]), int(e[1])))
        elif isinstance(e, (int, np.integer)) and default_grid is not None:
            out.add((default_grid, int(e)))
        else:
            raise ValueError(f'unrecognised split entry {e!r}')
    return out

def read_split(name):
    return json.load(open(f'{DATA_ROOT}/splits/{name}'))

base = read_split('indices.json')
splits = {r: {p for p in to_pairs(base[r], 25) if p[0] == 25} for r in ('train', 'val', 'test')}
splits['pool'] = {p for p in to_pairs(read_split('finetune_pool_indices.json')) if p[0] in FT_GRIDS}
splits['val'] |= {p for p in to_pairs(read_split('finetune_val_indices.json')) if p[0] in FT_GRIDS}
splits['test'] |= {p for p in to_pairs(read_split('finetune_test_indices.json')) if p[0] in FT_GRIDS}
assert not any(g == 55 for v in splits.values() for g, _ in v)

roles = list(splits)
for a in roles:
    for b in roles:
        if a < b:
            assert not (splits[a] & splits[b]), f'STOP: split overlap {a}/{b}: {len(splits[a] & splits[b])}'

# The evaluation sets must be EXACTLY the ones Chunk 28/29 scored (so their numbers stay comparable)
ref = pd.read_parquet(f'{ART}/ch28_per_sample.parquet', columns=['model_id', 'split', 'grid_size', 'local_idx'])
ref = ref[(ref['model_id'] == 'best_model') & ref['grid_size'].isin(GRIDS)]
def ref_set(split, grids):
    r = ref[(ref['split'] == split) & ref['grid_size'].isin(grids)]
    return set(zip(r['grid_size'].astype(int), r['local_idx'].astype(int)))
for role, split, grids in (('test', '25x25_test', (25,)), ('val', '25x25_val', (25,)),
                           ('test', 'finetune_test', FT_GRIDS), ('val', 'finetune_val', FT_GRIDS)):
    mine = {p for p in splits[role] if p[0] in grids}
    theirs = ref_set(split, grids)
    assert mine == theirs, (f'STOP: {role} {grids} differs from Chunk 28 {split}: '
                            f'{len(mine - theirs)} only here, {len(theirs - mine)} only in Chunk 28')
print('Val and test sets match the Chunk 28/29 evaluation sets exactly (grid 55 excluded).')

json.dump({r: sorted(map(list, v)) for r, v in splits.items()}, open(OUT('ch30_splits.json'), 'w'), indent=1)
for r, v in splits.items():
    print(f'{r:<6}', {g: sum(x[0] == g for x in v) for g in GRIDS})

def load_splits():
    d = json.load(open(f'{ART}/ch30_splits.json'))
    return {r: {tuple(p) for p in v} for r, v in d.items()}

## Step 2a — Feed-component transform
`src/feed_component.py` appends `x[:, 5]` = membership of the feed's **8-connected** metal component, with the virtual node set to 0. It is appended last, so `is_seed` stays at column 3.

The check is **independent**: on all 1,500 graphs in `ch29_eigs_sample.parquet`, the mask size must equal Chunk 29 Cell E's `n_nodes`, columns 0–4 must be bit-identical, and the virtual node must be 0. v1 compared the function against a copy of itself.

In [ ]:
# ── Step 2a: feed-component transform, checked against Chunk 29's independent counts ──
eig29 = pd.read_parquet(f'{ART}/ch29_eigs_sample.parquet')
assert set(eig29['grid'].unique()) <= set(GRIDS)
bad = 0
for _, r in tqdm(eig29.iterrows(), total=len(eig29), desc='feed-component check', leave=False):
    g, i = int(r['grid']), int(r['local_idx'])
    d = load_raw(g, i)
    x0 = d.x.clone()
    pat = d.x[:-1, 0].numpy().reshape(g, g)
    m = feed_component_mask(pat, g)
    e = TRANSFORM(Data(x=d.x.clone(), edge_index=d.edge_index, edge_attr=d.edge_attr))
    ok = (int(m.sum()) == int(r['n_nodes'])
          and e.x.shape == (g * g + 1, 6)
          and torch.equal(e.x[:, :5], x0)
          and float(e.x[-1, 5]) == 0.0 and float(e.x[-1, 3]) == -1.0
          and int(e.x[:-1, 5].sum()) == int(r['n_nodes']))
    bad += not ok
assert bad == 0, f'STOP: {bad} of {len(eig29)} graphs disagree with Chunk 29 Cell E n_nodes or break the column contract'
print(f'Feed-component transform matches Chunk 29 Cell E on all {len(eig29)} graphs; '
      f'columns 0-4 unchanged, virtual node 0.')
print(f'Feed component <= 21 nodes in {(eig29["n_nodes"] <= 21).mean()*100:.1f}% of these graphs.')

## Step 2b — Zero-column extension and exactness guard
**What it does:**
- Loads `best_model.pt` into `AntennaGNN(node_feat_dim=5, …)`, asserting 587,001 parameters.
- Asserts the only 5-input `Linear` is `input_proj`. This is confirmed in `src/model.py`, where `data.x` is consumed only by `input_proj` and by the masks on columns 0 and 3.
- Copies every weight into a 6-input model (587,129 parameters) with column 5 of `input_proj.weight` set to zero.

**Exactness guard:** 100 graphs per grid, compared in **float64 on CPU**, must agree to < 1e-5 dB. GPU GATv2 scatter is non-deterministic: Chunk 29 measured 1e-4 dB run-to-run, which would make a GPU comparison fail for reasons unrelated to the extension.

In [ ]:
# ── Step 2b: extend best_model to 6 inputs with a zero column; prove exactness ──
ck = torch.load(f'{CKPT}/best_model.pt', map_location='cpu', weights_only=False)
state = ck['model_state'] if isinstance(ck, dict) and 'model_state' in ck else ck
old = build_gnn(5); old.load_state_dict(state, strict=True)
assert n_params(old) == 587_001, n_params(old)

hits = [n for n, m in old.named_modules() if isinstance(m, nn.Linear) and m.in_features == 5]
assert hits == ['input_proj'], f'STOP: expected exactly one 5-input Linear named input_proj, got {hits}'
W_KEY = 'input_proj.weight'

new = build_gnn(6)
ns = new.state_dict()
for k, v in state.items():
    if k == W_KEY:
        assert ns[k].shape == (128, 6) and v.shape == (128, 5)
        w = torch.zeros(128, 6, dtype=v.dtype); w[:, :5] = v
        ns[k] = w
    else:
        assert ns[k].shape == v.shape, k
        ns[k] = v.clone()
new.load_state_dict(ns, strict=True)
assert n_params(new) == 587_001 + 128, n_params(new)
assert float(new.input_proj.weight[:, 5].abs().max()) == 0.0

# Exactness guard on CPU in float64: GPU GATv2 scatter is non-deterministic (Chunk 29 saw
# 1e-4 dB run-to-run), so a GPU comparison could fail spuriously. float64 on CPU isolates the
# algebra: a zero column must change nothing.
old64, new64 = copy.deepcopy(old).double().eval(), copy.deepcopy(new).double().eval()
STD64 = S11_STD.double()
rngX = np.random.RandomState(302)
worst = 0.0
for g in GRIDS:
    pool_g = sorted(p for p in splits['train'] | splits['pool'] | splits['val'] if p[0] == g)
    pick = [pool_g[j] for j in rngX.choice(len(pool_g), min(100, len(pool_g)), replace=False)]
    a_list, b_list = [], []
    for _, i in pick:
        d = load_raw(g, i)
        a_list.append(Data(x=d.x.double(), edge_index=d.edge_index, edge_attr=d.edge_attr.double()))
        b = TRANSFORM(Data(x=d.x.clone(), edge_index=d.edge_index, edge_attr=d.edge_attr))
        b_list.append(Data(x=b.x.double(), edge_index=b.edge_index, edge_attr=b.edge_attr.double()))
    with torch.no_grad():
        for A, B in zip(DataLoader(a_list, batch_size=25), DataLoader(b_list, batch_size=25)):
            d_db = ((old64(A) - new64(B)) * STD64).abs().max().item()
            worst = max(worst, d_db)
    print(f'  {g}x{g}: max |old - new| so far {worst:.3e} dB')
assert worst < 1e-5, f'STOP: zero-column extension changes the output by {worst:.3e} dB'
print(f'Exactness guard passed: max difference {worst:.3e} dB over {3*100} graphs (float64, CPU).')
del old64, new64

## Step 2c — Legacy fine-tune on the 35+45 pool
**Recipe:** Chunk 22 arm L0 (the Chunk 21–27 standard), read from the repo:
- normalized 201-point MSE
- Adam, lr 1e-4, weight decay 1e-4
- `ReduceLROnPlateau(0.5, 5, min 1e-6)`
- at most 60 epochs; early stop with patience 10 on val loss
- gradient clipping at 1.0; seed 42; batch 128

**Data:** `FinetuneDataset` is copied verbatim (frozen, Chunk 13), followed by the feed-component transform. The data is preloaded into RAM.

**Mechanics:** a checkpoint is written every epoch with optimizer, scheduler and RNG state, and a re-run resumes from it. Epoch 1 is timed, and the run aborts if the worst-case projection exceeds 1.5 h.

**Writes:** `checkpoints/backbone_no55.pt`.

In [ ]:
# ── Step 2c: fine-tune the extended LEGACY model on the 35+45 no-55 pool ──
# Recipe = Chunk 22 arm L0 (the Chunk 21-27 standard): normalized 201-point MSE, Adam lr 1e-4,
# wd 1e-4, ReduceLROnPlateau(0.5, 5, min 1e-6), max 60 epochs, early stop patience 10 on
# val loss, clip 1.0, seed 42, batch 128.

# frozen — validated in Chunk 13 (FinetuneDataset, copied verbatim)
class FinetuneDataset(TorchDataset):
    """Fine-tune graph dataset with mandatory z-score normalization at load.

    On-disk data.y is raw dB. This dataset z-scores y at load time using
    the 25x25 training-split s11_mean / s11_std, and preserves y_raw.
    evaluate() de-normalizes with the same statistics.

    Never pass s11_mean=None or s11_std=None.
    """

    def __init__(self, indices, processed_dir_base, s11_mean, s11_std):
        assert s11_mean is not None, 'FinetuneDataset requires s11_mean (got None)'
        assert s11_std is not None,  'FinetuneDataset requires s11_std (got None)'
        self.indices = indices
        self.processed_dir_base = processed_dir_base
        self.s11_mean = s11_mean   # (201,) tensor
        self.s11_std  = s11_std    # (201,) tensor

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        grid_size, local_idx = self.indices[idx]
        path = f'{self.processed_dir_base}/{grid_size}x{grid_size}/sample_{local_idx}.pt'
        data = torch.load(path, weights_only=False)

        # Preserve raw dB target, then z-score
        data.y_raw = data.y.clone()                              # (1, 201) raw dB
        data.y = (data.y - self.s11_mean) / (self.s11_std + 1e-8)  # (1, 201) normalized

        return data

def make_items(indices):
    """FinetuneDataset items + the feed-component transform, rebuilt as clean Data objects
    (so staged and Drive graphs collate identically). Preloaded into RAM."""
    assert all(STAGED[g] for g, _ in indices), 'graphs must be staged locally for training'
    ds = FinetuneDataset(indices, LOCAL, S11_MEAN, S11_STD)
    items = []
    for k in tqdm(range(len(ds)), desc='load', leave=False):
        d = ds[k]
        e = TRANSFORM(Data(x=d.x, edge_index=d.edge_index, edge_attr=d.edge_attr))
        g, i = indices[k]
        items.append(Data(x=e.x, edge_index=e.edge_index, edge_attr=e.edge_attr, y=d.y, y_raw=d.y_raw,
                          grid_size=int(g), local_idx=int(i)))
    # normalization identity guards (never range checks on mean/std)
    s = items[0]
    assert not torch.equal(s.y, s.y_raw)
    assert (s.y * (S11_STD + 1e-8) + S11_MEAN - s.y_raw).abs().max() < 1e-3
    return items

pool_idx = sorted(splits['pool'])
val_ft_idx = sorted(p for p in splits['val'] if p[0] in FT_GRIDS)
POOL_ITEMS, VAL_ITEMS = make_items(pool_idx), make_items(val_ft_idx)
print(f'pool {len(POOL_ITEMS)} | val {len(VAL_ITEMS)} (35/45, no 55)')

@torch.no_grad()
def eval_db(model, items, bs=64):
    model.eval()
    rec, loss_sum, n = [], 0.0, 0
    for b in DataLoader(items, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        p = model(b)
        loss_sum += F.mse_loss(p, b.y.squeeze(1), reduction='sum').item() / 201; n += p.shape[0]
        pdb = (p * S11_STD.to(DEVICE) + S11_MEAN.to(DEVICE)).cpu().numpy()
        tdb = b.y_raw.squeeze(1).cpu().numpy()
        for k in range(len(pdb)):
            rec.append(dict(grid=int(b.grid_size[k]), local_idx=int(b.local_idx[k]),
                            mse=float(np.mean((pdb[k] - tdb[k]) ** 2)), mae=float(np.mean(np.abs(pdb[k] - tdb[k]))),
                            true_min=float(tdb[k].min()), pred_min=float(pdb[k].min())))
    return pd.DataFrame(rec), loss_sum / n

def summarize(df):
    out = {}
    for g, s in list(df.groupby('grid')) + [('POOL', df)]:
        y = (s['true_min'] < -10).values
        out[g] = dict(n=len(s), mse=s['mse'].mean(), mae=s['mae'].mean(),
                      auroc=roc_auc_score(y, -s['pred_min']) if 0 < y.sum() < len(y) else np.nan,
                      acc=((s['pred_min'] < -10).values == y).mean())
    return out

MAX_EPOCHS, PATIENCE, MAX_NORM, LR, WD, BS, SEED = 60, 10, 1.0, 1e-4, 1e-4, 128, 42
RUN = f'{CKPT}/ch30_backbone_no55_running.pt'
FINAL = f'{CKPT}/backbone_no55.pt'
assert HAS_GPU, 'Step 2c needs the GPU'

torch.manual_seed(SEED); np.random.seed(SEED)
model = copy.deepcopy(new).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, min_lr=1e-6)
gen = torch.Generator().manual_seed(SEED)
start_ep, best_loss, best_state, best_ep, bad_ep, hist = 0, float('inf'), None, -1, 0, []
if os.path.exists(RUN):
    r = torch.load(RUN, map_location='cpu', weights_only=False)
    model.load_state_dict(r['model']); opt.load_state_dict(r['opt']); sched.load_state_dict(r['sched'])
    gen.set_state(r['gen']); torch.set_rng_state(r['torch_rng']); np.random.set_state(r['np_rng'])
    if HAS_GPU and r.get('cuda_rng') is not None: torch.cuda.set_rng_state_all(r['cuda_rng'])
    start_ep, best_loss, best_state, best_ep, bad_ep, hist = (r['epoch'] + 1, r['best_loss'], r['best_state'],
                                                             r['best_ep'], r['bad_ep'], r['hist'])
    print(f'Resumed after epoch {r["epoch"] + 1}')

loader = DataLoader(POOL_ITEMS, batch_size=BS, shuffle=True, num_workers=0, generator=gen)
t_run = time.time()
for ep in range(start_ep, MAX_EPOCHS):
    if bad_ep >= PATIENCE:
        break
    t0 = time.time(); model.train(); tl, nb = 0.0, 0
    for b in loader:
        b = b.to(DEVICE); opt.zero_grad()
        loss = F.mse_loss(model(b), b.y.squeeze(1))
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM); opt.step()
        tl += loss.item(); nb += 1
    vdf, vloss = eval_db(model, VAL_ITEMS)
    sched.step(vloss)
    vs = summarize(vdf)
    hist.append(dict(epoch=ep + 1, train_loss=tl / nb, val_loss=vloss, val_mse_db2=vs['POOL']['mse'],
                     lr=opt.param_groups[0]['lr'], sec=time.time() - t0))
    if vloss < best_loss:
        best_loss, best_ep, bad_ep = vloss, ep + 1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_ep += 1
    print(f'E{ep+1:2d} train {tl/nb:.4f} | val {vloss:.4f} | val MSE {vs["POOL"]["mse"]:.4f} dB^2 '
          f'(35: {vs[35]["mse"]:.3f}, 45: {vs[45]["mse"]:.3f}) | lr {opt.param_groups[0]["lr"]:.1e} | {time.time()-t0:.0f}s')
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), gen=gen.get_state(),
                    torch_rng=torch.get_rng_state(), np_rng=np.random.get_state(),
                    cuda_rng=torch.cuda.get_rng_state_all() if HAS_GPU else None, epoch=ep,
                    best_loss=best_loss, best_state=best_state, best_ep=best_ep, bad_ep=bad_ep, hist=hist), RUN)
    if ep == start_ep:
        proj_h = (time.time() - t0) * (MAX_EPOCHS - start_ep) / 3600
        print(f'   epoch time {time.time()-t0:.0f}s -> worst-case projection {proj_h:.2f} h')
        assert proj_h <= 1.5, f'STOP: projected {proj_h:.2f} h > 1.5 h allowance (checkpoint saved; resume later)'

model.load_state_dict(best_state)
torch.save({'model_state': best_state, 'node_feat_dim': 6, 'best_epoch': best_ep, 'best_val_loss': best_loss,
            'hist': hist, 'provenance': {'warm_start': 'best_model.pt (input_proj extended with a zero column)',
                                         'train_pool_grids': [35, 45], 'grid_55': 'excluded',
                                         'recipe': 'Chunk 22 L0', 'feature_5': 'feed 8-connected component'}},
           FINAL)
print(f'Saved {FINAL}: best epoch {best_ep}, val loss {best_loss:.4f}, total {(time.time()-t_run)/60:.1f} min this session')

## Step 2d — Pre-registered gate
**Gate:** `POOL_FINETUNE_VAL` MSE ≤ **1.35 dB²**, which is `ceiling_fullpool`'s no-55 value of 1.2313 plus 10% for the smaller pool.

**Also reports:**
- per-grid MSE, MAE and AUROC next to `ceiling_fullpool`'s
- 25×25 val MSE next to `best_model`'s 1.3129 (the forgetting check)
- `|W[:, 5]|` against the other input columns, showing whether the feature was used

**On FAIL:** run Steps 3 and 5, skip Step 4, then Chunk 30R.

**Writes:** `ch30_backbone_report.json`, including the checkpoint's SHA-256.

In [ ]:
# ── Step 2d: pre-registered gate ──
GATE_DB2 = 1.35   # ceiling_fullpool restated without 55 = 1.2313 (POOL_FINETUNE_VAL), +10%
vdf, _ = eval_db(model, VAL_ITEMS)
vs = summarize(vdf)
val25_idx = sorted(p for p in splits['val'] if p[0] == 25)
v25df, _ = eval_db(model, make_items(val25_idx))
v25 = summarize(v25df)[25]

rest = pd.read_csv(f'{ART}/ch29_no55_restatement.csv')
def ref_val(model_id, scope, col='mse'):
    r = rest[(rest['model_id'] == model_id) & (rest['variant'] == 'no55') & (rest['scope'] == scope)]
    return float(r[col].iloc[0]) if len(r) else float('nan')

rows = []
for scope, key in (('grid_35_VAL', 35), ('grid_45_VAL', 45), ('POOL_FINETUNE_VAL', 'POOL')):
    m = vs[key]
    rows.append(dict(scope=scope, backbone_no55_mse=m['mse'], ceiling_mse=ref_val('ceiling_fullpool', scope),
                     backbone_no55_mae=m['mae'], ceiling_mae=ref_val('ceiling_fullpool', scope, 'mae'),
                     backbone_no55_auroc=m['auroc'], ceiling_auroc=ref_val('ceiling_fullpool', scope, 'auroc')))
rows.append(dict(scope='grid_25_VAL', backbone_no55_mse=v25['mse'], ceiling_mse=ref_val('best_model', 'grid_25_VAL'),
                 backbone_no55_mae=v25['mae'], ceiling_mae=ref_val('best_model', 'grid_25_VAL', 'mae'),
                 backbone_no55_auroc=v25['auroc'], ceiling_auroc=ref_val('best_model', 'grid_25_VAL', 'auroc')))
REP = pd.DataFrame(rows)
print('(grid_25_VAL reference column is best_model, not ceiling_fullpool)')
print(REP.round(4).to_string(index=False))

w = model.input_proj.weight.detach().cpu()
w5, wother = float(w[:, 5].norm()), float(w[:, :5].norm(dim=0).mean())
print(f'\n|W[:, 5]| = {w5:.4f} vs mean |W[:, j<5]| = {wother:.4f} '
      f'({"feature used" if w5 > 0.05 * wother else "feature barely used"})')

pool_mse = vs['POOL']['mse']
GATE_PASS = bool(pool_mse <= GATE_DB2)
print(f'\nPRE-REGISTERED GATE: POOL_FINETUNE_VAL MSE {pool_mse:.4f} dB^2 <= {GATE_DB2} -> '
      f'{"PASS" if GATE_PASS else "FAIL"}')
if not GATE_PASS:
    print('FAIL -> run Steps 3 and 5, SKIP Step 4, then Chunk 30R (diagonal edges).')
json.dump(dict(gate_db2=GATE_DB2, pool_finetune_val_mse=pool_mse, gate_pass=GATE_PASS,
               per_scope=REP.to_dict(orient='records'), w5_norm=w5, w_other_mean_norm=wother,
               best_epoch=best_ep, checkpoint=FINAL, checkpoint_sha=file_sha(FINAL)),
          open(OUT('ch30_backbone_report.json'), 'w'), indent=2, default=float)

## Step 3 — K = 3 mode targets (CPU)
**(a) Reproduction.** Refits 300 of Chunk 29's sample curves with `src/synth.py` and must reproduce Chunk 29:
- fit MSE within max(10%, 0.005 dB²) for ≥ 95% of curves
- active f0 within 2 MHz for ≥ 95% of active modes
- K = 3 worse than K = 2 by more than 0.01 dB² for ≤ 1% of curves

The optimisation path is identical to Chunk 29's (verified bit-exact offline), so any failure means the curve source differs.

**(b) Production.** Fits 25×25 train/val and 35/45 pool/val in 1,000-curve shards, 35/45 first. The cell is resumable and projects from **measured** time. Test curves are never fitted. `depth_db` uses the true sigmoid baseline; v1 used `1/(1+exp(+poly))`, which is 1 − sigmoid. The mode-loss exclusion is `fit_mse > 0.17` **only**; `converged` is recorded and never used.

**Writes:** `ch30_mode_targets_ft.parquet` (35/45; enough to start Chunk 31) and `ch30_mode_targets.parquet` (everything).

In [ ]:
# ── Step 3: K = 3 mode targets (scipy, sharded, resumable; CPU-only) ──
# Standalone: needs only the header, Setup, Staging and Step 1 (or ch30_splits.json).
if 'splits' not in globals():
    splits = load_splits()
assert K_SLOTS == 3 and BASE_ORDER == 2

def raw_y(g, i):
    y = load_raw(g, i).y.reshape(-1).numpy().astype(np.float64)
    assert y.shape == (201,) and y.max() <= 0.01, f'{g}x{g} sample_{i}: y is not raw dB'
    return y

# ---- (a) reproduction check against Chunk 29 (same code path, so it should be exact) ----
fit29 = pd.read_parquet(f'{ART}/ch29_fit_stats_sample.parquet')
par29 = pd.read_parquet(f'{ART}/ch29_resonator_params_sample.parquet')
k3 = fit29[(fit29['variant'] == 'K3q') & fit29['ok']].set_index('curve_id')
k2 = fit29[(fit29['variant'] == 'K2q') & fit29['ok']].set_index('curve_id')
val_ids = np.random.RandomState(303).choice(k3.index.values, 300, replace=False)

def _fit_one(args):
    cid, g, i = args
    r = fit_curve_scipy(raw_y(g, i), K=3, order=2)
    return cid, r['fit_mse'], mode_rows_from_fit(r), r['time_s']

jobs = [(int(c), int(k3.loc[c, 'grid']), int(k3.loc[c, 'local_idx'])) for c in val_ids]
with mp.get_context('fork').Pool(N_CPU, initializer=_worker_init) as pool:
    res = pool.map(_fit_one, jobs, chunksize=4)
mse_ok, f0_ok, f0_n, nest_bad, times = 0, 0, 0, 0, []
for cid, mse, modes, ts in res:
    times.append(ts)
    ref_mse = float(k3.loc[cid, 'mse_fl'])
    mse_ok += mse <= ref_mse + max(0.10 * ref_mse, 0.005)
    nest_bad += mse > float(k2.loc[cid, 'mse_fl']) + 0.01
    ref_act = par29[(par29['curve_id'] == cid) & (par29['variant'] == 'K3q') & (par29['s'] >= 0.5) & (par29['min_db_fit'] < -3)]
    mine = [m for m in modes if m['s'] >= 0.5 and m['depth_db'] < -3]
    for f in ref_act['f0_GHz']:
        f0_n += 1
        f0_ok += bool(mine) and min(abs(m['f0'] - f) for m in mine) <= 0.002
print(f'Reproduction vs Chunk 29 (300 curves): fit MSE within tolerance {mse_ok/300*100:.1f}%, '
      f'active f0 within 2 MHz {f0_ok/max(f0_n,1)*100:.1f}% of {f0_n}, '
      f'K3 worse than K2 by > 0.01: {nest_bad/300*100:.2f}%')
assert mse_ok / 300 >= 0.95 and f0_ok / max(f0_n, 1) >= 0.95 and nest_bad / 300 <= 0.01, 'STOP: reproduction failed'

# ---- (b) production: ~100k curves, sharded, fine-tune grids first ----
GROUPS = [('pool', FT_GRIDS), ('val', FT_GRIDS), ('val', (25,)), ('train', (25,))]
SHARD = 1000
SHARD_DIR = f'{ART}/ch30_mode_shards'
os.makedirs(SHARD_DIR, exist_ok=True)
plan = []
for role, grids in GROUPS:
    keys = sorted(p for p in splits[role] if p[0] in grids)
    tag = f"{role}_{'_'.join(map(str, grids))}"
    for s in range(0, len(keys), SHARD):
        plan.append((tag, role, s // SHARD, keys[s:s + SHARD]))
assert not any(role == 'test' for _, role, _, _ in plan), 'test curves are never fitted'
n_total = sum(len(k) for *_, k in plan)
t_med = float(np.median(times))
print(f'{n_total:,} curves in {len(plan)} shards | measured {t_med:.2f} s/fit (median) on {N_CPU} CPUs '
      f'-> projected {n_total * np.mean(times) / N_CPU / 3600:.1f} h total; '
      f'fine-tune grids first ({sum(len(k) for t,_,_,k in plan if t.startswith("pool") or t=="val_35_45"):,} curves).')
if N_CPU < 12:
    print('   NOTE: fewer than 12 CPUs. Shards are resumable: you can stop, and continue in another session.')

def _fit_target(args):
    role, g, i = args
    r = fit_curve_scipy(raw_y(g, i), K=3, order=2)
    if not r['ok']:
        return []
    return [dict(sample_id=int(i), grid=int(g), split=role, **m, fit_mse=r['fit_mse'],
                 fit_mse_raw=r['fit_mse_raw'], converged=r['converged']) for m in mode_rows_from_fit(r)]

for tag, role, si, keys in tqdm(plan, desc='shards'):
    path = f'{SHARD_DIR}/{tag}_{si:04d}.parquet'
    if os.path.exists(path) and pd.read_parquet(path, columns=['sample_id'])['sample_id'].nunique() == len(keys):
        continue
    with mp.get_context('fork').Pool(N_CPU, initializer=_worker_init) as pool:
        out = pool.map(_fit_target, [(role, g, i) for g, i in keys], chunksize=8)
    rows = [r for o in out for r in o]
    assert len(rows) == 3 * len(keys), f'{path}: {len(keys) - len(rows)//3} fits failed'
    pd.DataFrame(rows).to_parquet(path + '.tmp', index=False); os.replace(path + '.tmp', path)

def assemble(tags, name):
    files = [f for t in tags for f in sorted(glob.glob(f'{SHARD_DIR}/{t}_*.parquet'))]
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    df['mode_loss_excluded'] = df['fit_mse'] > 0.17     # by fit MSE ONLY; `converged` is never used
    assert not (df['split'] == 'test').any() and not (df['grid'] == 55).any()
    df.to_parquet(OUT(name), index=False)
    per = df.drop_duplicates(['grid', 'sample_id'])
    print(f'{name}: {len(per):,} curves | excluded from mode losses {per["mode_loss_excluded"].mean()*100:.2f}% '
          f'| median fit MSE {per["fit_mse"].median():.4f} dB^2 | hit eval cap {(~per["converged"]).mean()*100:.1f}%')
    return df
MODES_FT = assemble(['pool_35_45', 'val_35_45'], 'ch30_mode_targets_ft.parquet')
MODES = assemble(['pool_35_45', 'val_35_45', 'val_25', 'train_25'], 'ch30_mode_targets.parquet')

## Step 4 — Embedding cache (only if the gate passed)
**What it does:**
- Asserts the gate passed and the checkpoint hash is unchanged.
- Hooks the 256-d input to `output_mlp` (the readout) and the output of the final GATv2 layer (node embeddings, 128-d).
- Proves that `output_mlp(readout)` reproduces the prediction.
- Stores node embeddings as float16 (~4.5 GB) in a memmap with offsets, with the virtual node last per graph, alongside per-node feed distance in mm (virtual node = −1).
- Asserts the model state hash is unchanged afterwards.

**Writes:** `ch30_emb_index.parquet`, `ch30_readout.npy`, `ch30_node_emb_f16.dat`, `ch30_node_emb_meta.json`, `ch30_node_feed_dist_mm.npy`.

In [ ]:
# ── Step 4: embedding cache from backbone_no55.pt (35/45 pool + val). Only if the gate passed. ──
REPORT = json.load(open(f'{ART}/ch30_backbone_report.json'))
assert REPORT['gate_pass'], 'Step 2 gate FAILED: skip Step 4 and run Chunk 30R'
assert HAS_GPU
assert file_sha(f'{CKPT}/backbone_no55.pt') == REPORT['checkpoint_sha'], 'checkpoint changed since the gate'
bb = build_gnn(6)
bb.load_state_dict(torch.load(f'{CKPT}/backbone_no55.pt', map_location='cpu', weights_only=False)['model_state'], strict=True)
bb = bb.to(DEVICE).eval()

def state_hash(m):
    h = hashlib.sha256()
    for k, v in sorted(m.state_dict().items()):
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()
H0 = state_hash(bb)

cache_keys = [('pool', p) for p in sorted(splits['pool'])] + \
             [('val', p) for p in sorted(p for p in splits['val'] if p[0] in FT_GRIDS)]
n_nodes = [g * g + 1 for _, (g, _) in cache_keys]
offsets = np.concatenate([[0], np.cumsum(n_nodes)])
LOCAL_EMB = '/content/ch30_node_emb_f16.dat'
emb = np.memmap(LOCAL_EMB, dtype=np.float16, mode='w+', shape=(int(offsets[-1]), 128))
dist = np.empty(int(offsets[-1]), dtype=np.float32)
readout = np.empty((len(cache_keys), 256), dtype=np.float32)

grab = {}
h1 = bb.output_mlp.register_forward_pre_hook(lambda m, inp: grab.__setitem__('r', inp[0].detach()))
h2 = bb.blocks[-1][-1].register_forward_hook(lambda m, inp, out: grab.__setitem__('h', out.detach()))
BSZ = 32
try:
    for s in tqdm(range(0, len(cache_keys), BSZ), desc='embed'):
        chunk = cache_keys[s:s + BSZ]
        items = []
        for role, (g, i) in chunk:
            d = load_raw(g, i)
            items.append(TRANSFORM(Data(x=d.x.clone(), edge_index=d.edge_index, edge_attr=d.edge_attr)))
        b = next(iter(DataLoader(items, batch_size=len(items)))).to(DEVICE)
        with torch.no_grad():
            pred = bb(b)
            # the cached readout must reproduce the prediction exactly through output_mlp
            assert (bb.output_mlp(grab['r']) - pred).abs().max().item() < 1e-5
        H = grab['h'].float().cpu().numpy()
        ptr = b.ptr.cpu().numpy()
        for k, (role, (g, i)) in enumerate(chunk):
            row = s + k
            assert ptr[k + 1] - ptr[k] == n_nodes[row]
            emb[offsets[row]:offsets[row + 1]] = H[ptr[k]:ptr[k + 1]].astype(np.float16)
            dist[offsets[row]:offsets[row + 1] - 1] = feed_distance_mm(g)
            dist[offsets[row + 1] - 1] = -1.0          # virtual node
        readout[s:s + len(chunk)] = grab['r'].cpu().numpy()
finally:
    h1.remove(); h2.remove()
emb.flush()
assert state_hash(bb) == H0, 'backbone state changed during extraction'

idx = pd.DataFrame([dict(row=r, split=role, grid=g, local_idx=i, node_offset=int(offsets[r]), n_nodes=n_nodes[r])
                    for r, (role, (g, i)) in enumerate(cache_keys)])
idx.to_parquet(OUT('ch30_emb_index.parquet'), index=False)
np.save(OUT('ch30_readout.npy'), readout)
np.save(OUT('ch30_node_feed_dist_mm.npy'), dist)
shutil.copy(LOCAL_EMB, OUT('ch30_node_emb_f16.dat'))
json.dump(dict(dtype='float16', shape=[int(offsets[-1]), 128], backbone_sha=REPORT['checkpoint_sha'],
               state_hash=H0, note='rows follow ch30_emb_index.parquet node_offset; virtual node is last per graph'),
          open(OUT('ch30_node_emb_meta.json'), 'w'), indent=2)
print(f'Cached {len(cache_keys):,} graphs, {int(offsets[-1]):,} node rows '
      f'({os.path.getsize(LOCAL_EMB)/1e9:.2f} GB float16); state hash unchanged.')

## Step 5 — V2 eigenvalue cache (CPU)
**What it does:**
- Uses `src/spectral.py`: the 8-connected feed component, the normalized Laplacian with diagonal edges × N², and the 20 smallest nonzero eigenvalues. It asserts the zero eigenvalue and returns `eig_valid = 0` when the component has ≤ 21 nodes.
- First reproduces Chunk 29's 1,500 V2 rows to a relative error < 1e-6.
- Then runs every split, test included, since these are features rather than labels, in resumable shards.

**Writes:** `ch30_eigs.parquet`.

In [ ]:
# ── Step 5: V2 eigen cache for every graph in every split (features only; CPU-only) ──
if 'splits' not in globals():
    splits = load_splits()

# (a) reproduce Chunk 29 Cell E v2's 1,500 rows first
e29 = pd.read_parquet(f'{ART}/ch29_eigs_sample.parquet')
def _eig_job(args):
    role, g, i = args
    pat = load_raw(g, i).x[:-1, 0].numpy().reshape(g, g)
    return dict(split=role, grid=int(g), local_idx=int(i), **eig_v2_features(pat, g))
with mp.get_context('fork').Pool(N_CPU, initializer=_worker_init) as pool:
    rep = pd.DataFrame(pool.map(_eig_job, [('chk', int(r.grid), int(r.local_idx)) for r in e29.itertuples()], chunksize=8))
assert (rep['n_nodes'].values == e29['n_nodes'].values).all() and (rep['eig_valid'].values == e29['eig_valid'].values).all()
v = e29['eig_valid'].values == 1
ours, theirs = rep.loc[v, EIG_COLS].values, e29.loc[v, [f'V2_{k}' for k in range(M_EIG)]].values
rel = np.abs(ours - theirs) / np.maximum(np.abs(theirs), 1e-12)
assert rel.max() < 1e-6, f'STOP: V2 eigenvalues differ from Chunk 29 by up to {rel.max():.2e} (relative)'
print(f'Reproduced Chunk 29 V2 on {len(e29)} graphs: max relative difference {rel.max():.2e}')

# (b) all splits, sharded and resumable
EIG_DIR = f'{ART}/ch30_eig_shards'
os.makedirs(EIG_DIR, exist_ok=True)
keys = [(role, g, i) for role in ('train', 'val', 'test', 'pool') for g, i in sorted(splits[role])]
assert not any(g == 55 for _, g, _ in keys)
for s in tqdm(range(0, len(keys), 5000), desc='eig shards'):
    path = f'{EIG_DIR}/{s // 5000:04d}.parquet'
    part = keys[s:s + 5000]
    if os.path.exists(path) and len(pd.read_parquet(path, columns=['local_idx'])) == len(part):
        continue
    with mp.get_context('fork').Pool(N_CPU, initializer=_worker_init) as pool:
        rows = pool.map(_eig_job, part, chunksize=16)
    pd.DataFrame(rows).to_parquet(path + '.tmp', index=False); os.replace(path + '.tmp', path)
EIGS = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f'{EIG_DIR}/*.parquet'))], ignore_index=True)
assert len(EIGS) == len(keys) and not (EIGS['grid'] == 55).any()
EIGS.to_parquet(OUT('ch30_eigs.parquet'), index=False)
print(f'ch30_eigs.parquet: {len(EIGS):,} graphs | eig_valid = 0 for '
      + ', '.join(f'{g}: {(EIGS[EIGS.grid == g].eig_valid == 0).mean()*100:.1f}%' for g in GRIDS))

## Final guard
**What it checks:**
- every artifact is named `ch30_*`
- there are no grid-55 rows
- the mode targets contain no test curves
- the s11 statistics are byte-identical
- no `ch28`/`ch29` artifact was modified
- the backbone is unchanged since the gate

**Writes:** `ch30_decisions.json` (gate result, artifact map, next chunk).

In [ ]:
# ── Final guard ──
fails = []
def guard(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok: fails.append(msg)

guard(all(os.path.basename(p).startswith('ch30_') for p in WRITTEN), f'all {len(WRITTEN)} artifacts are ch30_*')
for p in WRITTEN:
    if p.endswith('.parquet') and os.path.exists(p):
        t = pd.read_parquet(p)
        for c in ('grid', 'grid_size'):
            if c in t.columns:
                guard(not (t[c] == 55).any(), f'no grid-55 rows in {os.path.basename(p)}')
for name in ('ch30_mode_targets.parquet', 'ch30_mode_targets_ft.parquet'):
    p = f'{ART}/{name}'
    if os.path.exists(p):
        guard(not (pd.read_parquet(p, columns=['split'])['split'] == 'test').any(), f'{name}: no test curves')
guard(all(file_sha(f'{ART}/{n}') == h for n, h in S11_SHA.items()), 's11_mean/s11_std unchanged')
guard(all(os.path.getmtime(p) == t for p, t in PRIOR_MTIMES.items()), f'no ch28/ch29 artifact modified ({len(PRIOR_MTIMES)})')
if os.path.exists(f'{ART}/ch30_backbone_report.json'):
    rep = json.load(open(f'{ART}/ch30_backbone_report.json'))
    guard(file_sha(rep['checkpoint']) == rep['checkpoint_sha'], 'backbone_no55.pt unchanged since the gate')
    gate = rep['gate_pass']
else:
    gate = None
json.dump(dict(gate_pass=gate, backbone='checkpoints/backbone_no55.pt', node_feat_dim=6, feature_5='feed 8-connected component',
               mode_targets='ch30_mode_targets.parquet', mode_targets_ft='ch30_mode_targets_ft.parquet',
               eigs='ch30_eigs.parquet', embeddings='ch30_node_emb_f16.dat + ch30_emb_index.parquet',
               mode_loss_exclusion='fit_mse > 0.17 only', next='chunk31' if gate else 'chunk30R'),
          open(OUT('ch30_decisions.json'), 'w'), indent=2)
assert not fails, f'{len(fails)} guard(s) failed: {fails}'
print('ALL GUARDS PASSED. Next:', 'Chunk 31' if gate else 'Chunk 30R (diagonal edges)')